# ==============================================================================
# 🚢 MARITIME SURVIVAL: TITANIC VIA ADABOOST CLASSIFIER (BASIC INTUITION)
# ==============================================================================
### Core Principles:
AdaBoost (Adaptive Boosting) transforms a collection of weak learners (Decision Stumps with `max_depth=1`) into a high-accuracy ensemble.
Each sequential stump focuses on the samples misclassified by previous rounds by updating sample weights:
$$w_i^{(m+1)} = w_i^{(m)} \exp(-\alpha_m y_i h_m(\mathbf{x}_i))$$
Where estimator voting power $\alpha_m = \frac{1}{2} \ln\left(\frac{1 - \epsilon_m}{\epsilon_m}\right)$.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Titanic Environment ready.")


✅ STEP 1: Titanic Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Titanic dataset (891 passenger records).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/titanic/Titanic-Dataset.csv'
df = pd.read_csv(data_path)

# Drop high-cardinality text attributes
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} passengers, {df.shape[1]} attributes")
print(f"Survival Rate:\n{df['survived'].value_counts(normalize=True).round(4)*100}%")
df.head(3)


📊 Dataset Shape: 891 passengers, 8 attributes
Survival Rate:
survived
0    61.62
1    38.38
Name: proportion, dtype: float64%


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `survived`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['survived'])
y = df['survived']

print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")


Missing attributes:
age         177
embarked      2
dtype: int64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Cohort: {X_train.shape[0]} passengers ({y_train.sum()} Survived)")
print(f"Test Cohort : {X_test.shape[0]} passengers ({y_test.sum()} Survived)")


Train Cohort: 712 passengers (273 Survived)
Test Cohort : 179 passengers (69 Survived)


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Median imputation for `age`, mode imputation for `embarked`, and One-Hot encoding for categoricals.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['sex', 'embarked']
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])
print("✅ STEP 7: Preprocessor ready.")


✅ STEP 7: Preprocessor ready.


## ⚠️ STEP 8: SINGLE DECISION STUMP VS ADABOOST DUEL
Demonstrating how boosting 50 weak stumps drastically beats a single 1-split tree.


In [6]:
# STEP 8: BASELINE DUEL
# Single Stump
pipe_stump = Pipeline([
    ('prep', preprocessor),
    ('stump', DecisionTreeClassifier(max_depth=1, random_state=42))
])
pipe_stump.fit(X_train, y_train)

# AdaBoost Ensemble of 50 Stumps
pipe_ada = Pipeline([
    ('prep', preprocessor),
    ('ada', AdaBoostClassifier(n_estimators=50, random_state=42))
])
pipe_ada.fit(X_train, y_train)

acc_stump = accuracy_score(y_test, pipe_stump.predict(X_test))
acc_ada = accuracy_score(y_test, pipe_ada.predict(X_test))

print("="*65)
print(f"Single Weak Stump (Depth 1) Accuracy: {acc_stump * 100:.2f}%")
print(f"AdaBoost (50 Stumps) Accuracy        : {acc_ada * 100:.2f}%")
print(f"Sequential Boosting Accuracy Gain    : +{(acc_ada - acc_stump)*100:.2f}%!")
print("="*65)


Single Weak Stump (Depth 1) Accuracy: 77.65%
AdaBoost (50 Stumps) Accuracy        : 78.21%
Sequential Boosting Accuracy Gain    : +0.56%!


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `n_estimators` and `learning_rate` shrinkage.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'ada__n_estimators': [50, 100, 150],
    'ada__learning_rate': [0.05, 0.1, 0.5, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_ada, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Accuracy: {grid.best_score_ * 100:.2f}%")


🏆 Best Hyperparameters: {'ada__learning_rate': 1.0, 'ada__n_estimators': 150}
🎯 Best 5-Fold CV Accuracy: 80.90%


## 📊 STEP 10 & 11: EVALUATION REPORT & CONFUSION MATRIX
Diagnostics on passenger survival prediction.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("📋 PASSENGER SURVIVAL CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Perished', 'Survived']))

print(f"🎯 Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 ROC-AUC Score: {roc_auc_score(y_test, y_prob)*100:.2f}%")
print(f"\nConfusion Matrix:\n{cm}")


📋 PASSENGER SURVIVAL CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Perished       0.79      0.87      0.83       110
    Survived       0.76      0.64      0.69        69

    accuracy                           0.78       179
   macro avg       0.78      0.76      0.76       179
weighted avg       0.78      0.78      0.78       179

🎯 Test Accuracy: 78.21%
🎯 ROC-AUC Score: 82.82%

Confusion Matrix:
[[96 14]
 [25 44]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying pipeline and validating instant boarding pass safety screening latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/titanic_adaboost_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_passenger = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_surv = loaded_pipe.predict(sample_passenger)[0]
pred_prob = loaded_pipe.predict_proba(sample_passenger)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚢 LIVE PASSENGER RISK AUDIT:")
print(f"   Survival Outcome   : {'🟢 SURVIVED' if pred_surv == 1 else '🔴 PERISHED'}")
print(f"   Survival Probability: {pred_prob*100:.2f}%")
print(f"   Inference Latency  : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/titanic_adaboost_pipeline.joblib (18,090 bytes)



🚢 LIVE PASSENGER RISK AUDIT:
   Survival Outcome   : 🔴 PERISHED
   Survival Probability: 37.85%
   Inference Latency  : 34.132 ms (SLA < 2.0ms: CHECK)
